https://www.kaggle.com/datasets/beekiran/sales-data-analysis?select=Sales+Data.csv

# Sales Data Analysis

**Data:** sales of an electronics store for 12 months of 2019 (laptops, phones, headphones, cables and more). Each row is **one product in one order**.

**Plan:** load the data, clean it (missing values, wrong rows, duplicates, outliers), then answer 5 questions with Plotly graphs.

| Column | Meaning |
|---|---|
| `Order ID` | Number of the order. One order can have several rows. |
| `Product` | What was bought |
| `Quantity Ordered` | How many pieces |
| `Price Each` | Price of one piece |
| `Order Date` | Date and time of the order |
| `Purchase Address` | Where the customer lives |
| `Unnamed: 0` | Old row number saved by mistake in the file (we will drop it) |
| `Month`, `Sales`, `City`, `Hour` | Extra columns already made in the file. We will make our own again. |

## Part 1: Load and look

In [2]:
import pandas as pd
import plotly.express as px

Download `Sales Data.csv` from Kaggle and upload it into Colab (folder icon on the left, then the upload button). Then read it.

In [8]:
df = pd.read_csv("Sales Data - Sales Data (1).csv")
df.shape

(185950, 11)

In [9]:
df.head()

,Unnamed: 0,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Month,Sales,City,Hour
0,0,295665,Macbook Pro Laptop,1,1700.00,2019-12-30 00:01:00,"136 Church St, New York City, NY 10001",12,1700.00,New York City,0
1,1,295666,LG Washing Machine,1,600.00,2019-12-29 07:03:00,"562 2nd St, New York City, NY 10001",12,600.00,New York City,7
2,2,295667,USB-C Charging Cable,1,11.95,2019-12-12 18:21:00,"277 Main St, New York City, NY 10001",12,11.95,New York City,18
3,3,295668,27in FHD Monitor,1,149.99,2019-12-22 15:13:00,"410 6th St, San Francisco, CA 94016",12,149.99,San Francisco,15
4,4,295669,USB-C Charging Cable,1,11.95,2019-12-18 12:38:00,"43 Hill St, Atlanta, GA 30301",12,11.95,Atlanta,12


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 185950 entries, 0 to 185949
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        185950 non-null  int64  
 1   Order ID          185950 non-null  int64  
 2   Product           185950 non-null  str    
 3   Quantity Ordered  185950 non-null  int64  
 4   Price Each        185950 non-null  float64
 5   Order Date        185950 non-null  str    
 6   Purchase Address  185950 non-null  str    
 7   Month             185950 non-null  int64  
 8   Sales             185950 non-null  float64
 9   City              185950 non-null  str    
 10  Hour              185950 non-null  int64  
dtypes: float64(2), int64(5), str(4)
memory usage: 15.6 MB


Look at the `Dtype` column. `Order Date` is stored as **text** (`object`), so we will turn it into a real date.

## Part 2: Clean the data

### The extra column

`Unnamed: 0` is just the old row number (0, 1, 2 ...) that was saved into the file. It is not data, and it would also stop duplicate rows from matching each other, so we drop it first.

In [11]:
df = df.drop(columns="Unnamed: 0", errors="ignore")
df.columns

Index(['Order ID', 'Product', 'Quantity Ordered', 'Price Each', 'Order Date',
       'Purchase Address', 'Month', 'Sales', 'City', 'Hour'],
      dtype='str')

### Missing values

In [12]:
df.isnull().sum()

Order ID            0
Product             0
Quantity Ordered    0
Price Each          0
Order Date          0
Purchase Address    0
Month               0
Sales               0
City                0
Hour                0
dtype: int64

The number next to each column is how many cells are empty. A sale with an empty product or price cannot be used, so we drop every row that has an empty cell. (If all numbers are 0, nothing is dropped and the code is still safe to run.)

In [13]:
df = df.dropna()
df.shape

(185950, 10)

Now we turn `Order Date` from text into a real date, so we can ask for the month, the hour and the year.

In [14]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df.dtypes

Order ID                     int64
Product                        str
Quantity Ordered             int64
Price Each                 float64
Order Date          datetime64[us]
Purchase Address               str
Month                        int64
Sales                      float64
City                           str
Hour                         int64
dtype: object

### Duplicates

A duplicate is a row that is exactly the same as another row in **every** column.

In [15]:
df.duplicated().sum()

np.int64(264)

In [16]:
df[df.duplicated(keep=False)].sort_values("Order ID").head(6)

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Month,Sales,City,Hour
137664,142071,AA Batteries (4-pack),1,3.84,2019-01-17 23:02:00,"131 2nd St, Boston, MA 02215",1,3.84,Boston,23
137665,142071,AA Batteries (4-pack),1,3.84,2019-01-17 23:02:00,"131 2nd St, Boston, MA 02215",1,3.84,Boston,23
140898,145143,Lightning Charging Cable,1,14.95,2019-01-06 03:01:00,"182 Jefferson St, San Francisco, CA 94016",1,14.95,San Francisco,3
140899,145143,Lightning Charging Cable,1,14.95,2019-01-06 03:01:00,"182 Jefferson St, San Francisco, CA 94016",1,14.95,San Francisco,3
142577,146765,Google Phone,1,600.00,2019-01-21 11:23:00,"918 Highland St, New York City, NY 10001",1,600.00,New York City,11
142578,146765,Google Phone,1,600.00,2019-01-21 11:23:00,"918 Highland St, New York City, NY 10001",1,600.00,New York City,11


If the number above is more than 0, these rows are exact copies: same order number, same product, same time and same address. Keeping them would count the same sale twice, so we keep the first one and drop the rest.

In [17]:
df = df.drop_duplicates()
df.shape

(185686, 10)

### Outliers

An outlier is a value that is very far from the others. We check three things.

**1. Dates.** All orders should be in the 12 months of 2019.

In [18]:
print(df["Order Date"].min())
print(df["Order Date"].max())
print("Rows in 2019:", (df["Order Date"].dt.year == 2019).sum())
print("Rows in 2020:", (df["Order Date"].dt.year == 2020).sum())

2019-01-01 03:07:00
2020-01-01 05:13:00
Rows in 2019: 185652
Rows in 2020: 34


Any order from **1 January 2020** is outside our 12 months. If we keep it, January would mix two different years, so we remove those rows.

In [19]:
df = df[df["Order Date"].dt.year == 2019]
df.shape

(185652, 10)

**2. Quantity.** How many pieces do people order?

In [21]:
px.histogram(df, x="Quantity Ordered", title="Quantity ordered in one row")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'Quantity Ordered=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'histogram',
              'x': {'bdata': ('AQEBAQEBAQIBBAIBAgIBAQEBAQEBAQ' ... 'EBAQECAQEBAQEBAQEBAQEBAQEBAQEB'),
                    'dtype': 'i1'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Quantity ordered in one row'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Quantity Ordered'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

Almost everyone buys 1 piece. A few buy 7, 8 or 9. Which product is it?

In [22]:
df[df["Quantity Ordered"] >= 7]["Product"].value_counts()

Product
AAA Batteries (4-pack)    30
AA Batteries (4-pack)      2
Name: count, dtype: int64

Only cheap AAA and AA batteries. Buying many batteries is normal, so these are **real** values and we keep them.

**3. Money per row.** First we make the `Sales` column (quantity times price), then draw a box plot.

In [23]:
df["Sales"] = df["Quantity Ordered"] * df["Price Each"]
px.box(df, y="Sales", title="Sales per row")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'alignmentgroup': 'True',
              'hovertemplate': 'Sales=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa'},
              'name': '',
              'notched': False,
              'offsetgroup': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'box',
              'x0': ' ',
              'xaxis': 'x',
              'y': {'bdata': ('AAAAAACQmkAAAAAAAMCCQGZmZmZm5i' ... 'ZmZmbmJ0BmZmZmZuYnQOxRuB6F6wdA'),
                    'dtype': 'f8'},
              'y0': ' ',
              'yaxis': 'y'}],
    'layout': {'boxmode': 'group',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Sales per row'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0]},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'Sales'}}}
})

Many dots sit above the box. Let's count them with the IQR rule (a value above `Q3 + 1.5 x IQR` is an outlier) and see which products they are.

In [24]:
q1 = df["Sales"].quantile(0.25)
q3 = df["Sales"].quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)

outliers = df[df["Sales"] > upper]
print("Upper limit:", upper, "| Outlier rows:", len(outliers))
outliers["Product"].value_counts()

Upper limit: 357.07500000000005 | Outlier rows: 36997


Product
iPhone                      6838
27in 4K Gaming Monitor      6224
34in Ultrawide Monitor      6174
Google Phone                5521
Macbook Pro Laptop          4720
ThinkPad Laptop             4124
Vareebadd Phone             2064
LG Washing Machine           666
LG Dryer                     646
Flatscreen TV                 19
Apple Airpods Headphones       1
Name: count, dtype: int64

The "outliers" are laptops, phones, monitors and TVs. They are not mistakes: **expensive products simply cost more.** The rule flags them only because most of the store's rows are cheap cables and batteries. We keep them.

> **Decision:** we dropped the extra column, the empty rows, the duplicate rows and the 2020 rows. We kept the batteries and the expensive products because they are real.

## Part 3: Answer 5 questions

The file already has `Month`, `Hour` and `City` columns, but we make them again so we know exactly how they are built. Two changes: our `Month` is the month **name** (so we can sort it), and our `City` also has the **state**, because there is a Portland in Maine and another Portland in Oregon, and the file's `City` mixes them.

In [25]:
df["Month"] = df["Order Date"].dt.month_name()
df["Hour"] = df["Order Date"].dt.hour

parts = df["Purchase Address"].str.split(",")
df["City"] = parts.str[1].str.strip() + " (" + parts.str[2].str.strip().str[:2] + ")"

### Question 1: Which month had the best sales?

In [27]:
months = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
monthly = df.groupby("Month")["Sales"].sum().reindex(months).reset_index()

px.bar(monthly, x="Month", y="Sales", title="Sales per month")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'Month=%{x}<br>Sales=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['January', 'February', 'March', 'April', 'May', 'June', 'July',
                          'August', 'September', 'October', 'November', 'December'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': ('7FG43gapO0GkcD0KB8lAQc3MzKx2Zk' ... 'oU7nx+TEFmZmaG2WVIQc3MzOxJlFFB'),
                    'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Sales per month'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Month'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'Sales'}}}
})

**December** is the best month (about 4.6 million), and **January** is the weakest (about 1.8 million). Sales also jump in October. Likely reasons are the holiday shopping season, but the data cannot prove that.

### Question 2: Which city sells the most?

In [ ]:
city_sales = df.groupby("City")["Sales"].sum().sort_values().reset_index()

px.bar(city_sales, x="Sales", y="City", orientation="h", title="Sales per city")

**San Francisco** is far ahead, and Los Angeles and New York City follow. Portland (Maine) is the smallest.

### Question 3: At what time of day do customers buy?

In [28]:
hourly = df.groupby("Hour")["Sales"].sum().reset_index()

px.line(hourly, x="Hour", y="Sales", markers=True, title="Sales by hour of the day")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'Hour=%{x}<br>Sales=%{y}<extra></extra>',
              'legendgroup': '',
              'line': {'color': '#636efa', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines+markers',
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('AAAAAAEAAAACAAAAAwAAAAQAAAAFAA' ... 'AAABMAAAAUAAAAFQAAABYAAAAXAAAA'),
                    'dtype': 'i4'},
              'xaxis': 'x',
              'y': {'bdata': ('XI/C9c+xJUFcj8L1S/gbQUjhehT0Yw' ... 'fhetYjP0GamZnZY4I4QR+F69HA+DFB'),
                    'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Sales by hour of the day'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Hour'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'Sales'}}}
})

There are two peaks: around **12 noon** and around **7 pm**. Ads shown just before these times would reach the most buyers.

### Question 4: Which products make the most money?

In [ ]:
product_sales = df.groupby("Product")["Sales"].sum().sort_values().reset_index()

px.bar(product_sales, x="Sales", y="Product", orientation="h", height=550, title="Sales per product")

The **MacBook Pro** earns the most, then the iPhone and the ThinkPad. Now compare with the number of pieces sold.

In [29]:
df.groupby("Product")["Quantity Ordered"].sum().sort_values(ascending=False).head(5)

Product
AAA Batteries (4-pack)      30981
AA Batteries (4-pack)       27615
USB-C Charging Cable        23927
Lightning Charging Cable    23163
Wired Headphones            20520
Name: Quantity Ordered, dtype: int64

The most **sold** products are cheap: charging cables and batteries. The most **profitable** are expensive: laptops and phones. Selling many cheap items and selling a few expensive ones are two different ways to earn money.

### Question 5: How much does a customer usually spend in one order?

One order can have several rows, so we add the rows of each `Order ID` together.

In [30]:
orders = df.groupby("Order ID")["Sales"].sum().reset_index()

px.histogram(orders, x="Sales", nbins=50, title="Total money in one order")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'Sales=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'nbinsx': 50,
              'orientation': 'v',
              'showlegend': False,
              'type': 'histogram',
              'x': {'bdata': ('AAAAAADghUBmZmZmZuYtQHsUrkfh+j' ... 'AAAHlAexSuR+H6J0CPwvUoXP9YQA=='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Total money in one order'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Sales'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

In [31]:
print("Typical order (median):", orders["Sales"].median())
print("Average order (mean):", round(orders["Sales"].mean(), 2))

Typical order (median): 14.95
Average order (mean): 193.14


Most orders are small (about 15), but a few are very large (laptops), which pulls the average up to about 190. So the **median** is the better description of a typical order.

## Summary

- Data cleaning removed empty rows, repeated header rows, duplicates and a few rows outside 2019.
- **December** is the best month and **San Francisco** the best city.
- Customers buy most around **12 noon and 7 pm**.
- **MacBook Pro** earns the most money, but cables and batteries are sold the most.
- A typical order is small, and a few big orders raise the average.

SyntaxError: invalid syntax (3727256512.py, line 1)